# 03 Random Forest - Tesla Market Direction

This is a complete, standalone pipeline for training a Random Forest classifier to predict whether Tesla stock will move Up or Down tomorrow.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

sns.set(style='whitegrid')

### 1. Data Preparation
Creating a binary movement target from the adjusted prices.

In [ ]:
def prepare_tesla_data(path):
    df = pd.read_csv(path)
    # Split adjustment
    df['adj_close'] = df['Close']
    for i in range(len(df)-1, 0, -1):
        if df.loc[i, 'Split_Factor'] > 1.0:
            df.loc[:i-1, 'adj_close'] /= df.loc[i, 'Split_Factor']
            
    # Target: 1 if next day close > current close, else 0
    df['direction'] = (df['adj_close'].shift(-1) > df['adj_close']).astype(int)
    
    features = ['RSI', 'Volatility_20d', 'Rel_Strength_SPY', 'SMA_50', 'Volume']
    df = df.dropna(subset=['direction'] + features)
    return df, features

df, feature_cols = prepare_tesla_data('../_data/stock_tesla.csv')
X = df[feature_cols]
y = df['direction']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

### 2. Model Training

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Random Forest Classifier trained.")

### 3. Evaluation & Visuals

In [ ]:
y_pred = model.predict(X_test)
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='YlOrRd')
plt.title('Tesla Directional Signal - Confusion Matrix')
plt.show()
print(classification_report(y_test, y_pred))

### 4. Model Saving

In [ ]:
joblib.dump(model, '../_model/tesla_trading_random_forest.joblib')
print("Model saved.")